In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q fastapi uvicorn langchain-google-genai langdetect pydantic nest-asyncio langchain-core

In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 91.8 MB/s eta 0:00:00


In [ ]:
import os
import nest_asyncio
from typing import Dict, List
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from langdetect import detect, DetectorFactory
# FIXED IMPORTS BELOW
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

In [ ]:
%%writefile app.py
import streamlit as st
import os
from langdetect import detect
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from google.colab import userdata

# 1. Page Configuration
st.set_page_config(page_title="GlobalMind AI", page_icon="🌐")
st.title("🌐 Multilingual Chatbot")

# 2. Initialize AI Model
if "llm" not in st.session_state:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    # Using your preferred model string
    st.session_state.llm = ChatGoogleGenerativeAI(model="models/gemini-1.5-flash")

# 3. Initialize Memory (Replaces chat_memory dict)
if "messages" not in st.session_state:
    st.session_state.messages = []

# 4. Helper for Cultural Rules
def get_etiquette(lang_code):
    etiquette = {
        "es": "Spanish: Warm/friendly. Use 'Usted' for formal.",
        "fr": "French: Polite. Use 'Vous'.",
        "de": "German: Direct/Professional. Use 'Sie'.",
        "hi": "Hindi: Respectful using 'Ji'.",
        "en": "English: Clear and helpful."
    }
    return etiquette.get(lang_code, "Respond naturally.")

# 5. Display Conversation History
for msg in st.session_state.messages:
    role = "user" if isinstance(msg, HumanMessage) else "assistant"
    with st.chat_message(role):
        st.markdown(msg.content)

# 6. User Input
if prompt := st.chat_input("Ask me anything..."):
    # Display user message
    st.chat_message("user").markdown(prompt)

    # Language Detection
    try:
        lang = detect(prompt)
    except:
        lang = "en"

    # AI Logic
    with st.chat_message("assistant"):
        with st.spinner(f"Processing ({lang})..."):
            rules = get_etiquette(lang)
            system_ins = SystemMessage(content=f"Multilingual AI. Lang: {lang}. Rules: {rules}")

            # Combine instructions + history + current message
            full_prompt = [system_ins] + st.session_state.messages[-6:] + [HumanMessage(content=prompt)]

            response = st.session_state.llm.invoke(full_prompt).content
            st.markdown(response)

    # Update Session State
    st.session_state.messages.append(HumanMessage(content=prompt))
    st.session_state.messages.append(AIMessage(content=response))

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

--2026-05-01 06:09:28--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-05-01 06:09:28--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-01T07%3A04%3A19Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-01T0

In [ ]:
# Kill everything first
!pkill streamlit
!pkill cloudflared

# Re-run Streamlit in the background
import subprocess
import time
with open("logs.txt", "w") as f:
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"], stdout=f, stderr=f)

# Give Streamlit 8 seconds to fully load the Gemini model
time.sleep(8)

# Start a NEW tunnel
!./cloudflared tunnel --url http://127.0.0.1:8501

2026-05-01T06:21:21Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-01T06:21:21Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-01T06:21:23Z INF +--------------------------------------------------------------------------------------------+
2026-05-01T06:21:23Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-01T06:21:23Z INF |  https://dame-everyday-goat-seen.trycloudflare.com    

In [ ]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))

print("Supported model names for your key:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"- {m.name}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Supported model names for your key:
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-3-1b-it
- models/gemma-3-4b-it
- models/gemma-3-12b-it
- models/gemma-3-27b-it
- models/gemma-3n-e4b-it
- models/gemma-3n-e2b-it
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3-pro-image-preview
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/lyria-3-clip-preview
- models/lyria-3-pro-preview
- models/gemini-3.1-flas